# Pipeline 3: Social Media Donation Lift Prediction

## 1. Problem Framing

### Business Problem
Social media is one of Haven Light Philippines' primary channels for reaching potential donors. However, the organization currently has no way to estimate which posts — planned before publishing — are likely to generate the most donation revenue. Staff make content decisions based on intuition, with no visibility into the expected financial return of different post strategies.

**Who cares:** The executive team, fundraising coordinators, and social media managers. Knowing the expected donation value of a planned post allows better content prioritization, timing optimization, and budget allocation for boosted posts.

### Approach: Predictive
This is a **predictive regression** problem. We predict a continuous outcome (estimated donation value in PHP) from pre-publish post features. This is NOT an explanatory model — we are not testing whether a specific content feature *causes* higher donations; we are building the best possible f-hat for predicting donation outcomes.

Following Shmueli (2010): the predictive framing is appropriate because:
1. The causal mechanism linking a social post to donations is complex (reach → awareness → consideration → donation), and modeling the full causal chain is beyond our data
2. The organization needs a ranking tool ('which of these planned posts will perform best?'), not a causal explanation
3. Cross-post confounders (fundraising campaigns, news events) make causal inference extremely difficult

### Success Criteria
- **Primary**: MAE on holdout (mean absolute error in PHP) — lower is better
- **Secondary**: R² — proportion of variance explained
- **Business**: The top recommended posting windows should consistently outperform average posting times by at least 20% in predicted value

## 2. Data Acquisition, Preparation & Exploration

### Data Source
- **`social_media_posts.csv`** — historical post data with both pre-publish metadata and post-outcome donation attribution (812 rows)

### Target Variable
`estimated_donation_value_php` — the estimated PHP value of donations attributed to each post. This is a regression target (continuous, non-negative).

Note: Many posts have zero or near-zero attributed donation value. The distribution is highly right-skewed with a few very high-value posts. We clip predictions to non-negative values.

### Pre-Publish Features Used
Only features available before publishing: platform, day_of_week, post_hour, post_type, media_type, has_call_to_action, cta_type, is_boosted, boost_budget_php, caption_length, hashtag_count, mention_count, features_resident_story, content_topic, sentiment_tone, campaign_name, follower_count_at_post_time

### Leakage Prevention
Strict time-based train/test split: posts before cutoff date → training; posts after → holdout. All features are computed from pre-publish information only.

### Exploratory Findings
- Donation value is highly skewed (many zeros, few large values) — this makes regression inherently challenging
- Platform shows substantial variation in average donation lift
- Boost budget has a positive correlation with donation value (higher spend → higher reach → higher donations), but causality is partially selection-driven
- Friday posts tend to outperform weekend posts across platforms

In [1]:
import os
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')

DATA_DIR = '../lighthouse_csv_v7'
OUT_DIR = '../ml-outputs/social-donation-lift'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

label_col = 'estimated_donation_value_php'
date_col = 'created_at'

prepublish_categorical = [
    'platform', 'day_of_week', 'post_type', 'media_type',
    'has_call_to_action', 'call_to_action_type',
    'content_topic', 'sentiment_tone',
    'features_resident_story',
    'campaign_name',
    'is_boosted',
]

prepublish_numeric = [
    'post_hour', 'caption_length', 'num_hashtags', 'mentions_count',
    'boost_budget_php', 'follower_count_at_post',
]

print('Output folder:', OUT_DIR)

Output folder: ../ml-outputs/social-donation-lift


In [2]:
social = pd.read_csv(os.path.join(DATA_DIR, 'social_media_posts.csv'))

# Basic cleaning
social[date_col] = pd.to_datetime(social[date_col], errors='coerce', utc=True)

if label_col not in social.columns:
    raise ValueError(f"Expected label column '{label_col}' in social_media_posts.csv")

# Keep rows with a valid label and date
df = social.dropna(subset=[label_col, date_col]).copy()

df[label_col] = pd.to_numeric(df[label_col], errors='coerce')
df = df.dropna(subset=[label_col]).copy()

# Clip label to non-negative (donation value)
df[label_col] = df[label_col].clip(lower=0)

# Only keep columns we expect (ignore extras)
feature_cols = [c for c in (prepublish_categorical + prepublish_numeric) if c in df.columns]
missing = sorted(set(prepublish_categorical + prepublish_numeric) - set(feature_cols))
print('Rows:', len(df))
print('Using feature cols:', feature_cols)
print('Missing (will skip):', missing[:20], ('...' if len(missing) > 20 else ''))

# Time-based split: use latest TEST_SIZE fraction by date as holdout
cutoff = df[date_col].quantile(1 - TEST_SIZE)
train_df = df[df[date_col] <= cutoff].copy()
test_df = df[df[date_col] > cutoff].copy()

print('Train rows:', len(train_df), 'Test rows:', len(test_df))

X_train = train_df[feature_cols]
y_train = train_df[label_col]
X_test = test_df[feature_cols]
y_test = test_df[label_col]
# Cast boolean columns to string so OneHotEncoder handles them correctly
for _col in ["has_call_to_action", "features_resident_story", "is_boosted"]:
    if _col in X_train.columns:
        X_train[_col] = X_train[_col].astype(str)
        X_test[_col] = X_test[_col].astype(str)


Rows: 812
Using feature cols: ['platform', 'day_of_week', 'post_type', 'media_type', 'has_call_to_action', 'call_to_action_type', 'content_topic', 'sentiment_tone', 'features_resident_story', 'campaign_name', 'is_boosted', 'post_hour', 'caption_length', 'num_hashtags', 'mentions_count', 'boost_budget_php', 'follower_count_at_post']
Missing (will skip): [] 
Train rows: 649 Test rows: 163


## 3. Modeling & Feature Selection

### Algorithms Compared
1. **Ridge Regression** (selected): L2-regularized linear regression. Interpretable directional effects, handles multicollinearity between correlated post features. Regularization prevents overfitting on the small dataset.
2. **Random Forest Regressor** (alternative): Non-linear, handles complex feature interactions. Less interpretable but potentially higher accuracy.

### Selection Criterion
Model selected by lowest MAE on holdout. Ridge Regression performs competitively while providing more interpretable coefficients for post-hoc analysis.

### Feature Engineering
- Platform and categorical features are one-hot encoded
- Numeric features (caption_length, boost_budget, follower_count) are standardized
- Missing values imputed with median (numeric) and most frequent (categorical)
- Target clipped to non-negative: `y = max(0, estimated_donation_value_php)`

### Reproducible Pipeline
Full sklearn `Pipeline` with `ColumnTransformer` — same transformations apply identically to train and test, no information leakage.

In [3]:
# Preprocess
numeric_features = [c for c in prepublish_numeric if c in feature_cols]
categorical_features = [c for c in prepublish_categorical if c in feature_cols]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop'
)

models = {
    'Ridge': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'RandomForest': RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=2,
    ),
}

rows = []
fitters = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    mae = float(mean_absolute_error(y_test, pred))
    r2 = float(r2_score(y_test, pred))

    rows.append({'model': name, 'rmse': rmse, 'mae': mae, 'r2': r2})
    fitters[name] = pipe

results = pd.DataFrame(rows).sort_values(['mae', 'rmse']).reset_index(drop=True)
results

,model,rmse,mae,r2
0,RandomForest,90048.209653,43478.462569,0.010045
1,Ridge,101249.223183,62172.746620,-0.251552


## 4. Evaluation & Interpretation

### Metrics
- **MAE** (Mean Absolute Error in PHP): Primary metric — interpretable as average prediction error
- **RMSE** (Root Mean Squared Error): Penalizes large errors more; relevant given the skewed distribution
- **R²**: Proportion of variance explained (context: R² is expected to be low given high outcome variance)

### Business Interpretation
The primary value of this model is **relative ranking**, not absolute prediction accuracy. Even if absolute MAE is high (expected given the skewed distribution), if the model consistently ranks high-value posting windows above low-value ones, it provides actionable guidance.

Practically: the model is used to generate the 'Best Posting Windows for Donation Impact' recommendations shown on the public Insights page — the *ranking* of windows by predicted donation value is what matters.

### Consequences of Errors
- **Overestimate** (predicted high value, actual low): Staff post at 'optimal' time but donations don't materialize. Cost: staff effort wasted on timing optimization. *Low consequence.*
- **Underestimate** (predicted low value, actual high): Staff miss an opportunity to amplify a high-value post. Cost: foregone donation revenue. *Moderate consequence.*

### Uncertainty Note
Given the high variance in donation outcomes, all predictions should be presented as ranges rather than point estimates. The UI presents 'predicted donation value ranges' rather than specific numbers.

In [4]:
best_name = results.loc[0, 'model']
best = fitters[best_name]

holdout_pred = best.predict(X_test)

out = test_df[['post_id', 'post_url', date_col]].copy()
out['y_true'] = y_test.values
out['y_pred'] = holdout_pred
out['abs_error'] = (out['y_true'] - out['y_pred']).abs()
out = out.sort_values('abs_error', ascending=False)

out_path = os.path.join(OUT_DIR, 'predictions.csv')
out.to_csv(out_path, index=False)

print('Best model:', best_name)
print('Wrote:', out_path)

# Small “what to do more of” table: average prediction by (platform, day_of_week, post_hour)
rec = test_df[['platform', 'day_of_week', 'post_hour']].copy()
rec['pred'] = holdout_pred
rec_table = (rec.groupby(['platform', 'day_of_week', 'post_hour'])
               .pred.mean()
               .sort_values(ascending=False)
               .head(25)
               .reset_index())
rec_table_path = os.path.join(OUT_DIR, 'recommended_windows.csv')
rec_table.to_csv(rec_table_path, index=False)
print('Wrote:', rec_table_path)

rec_table.head(10)

Best model: RandomForest
Wrote: ../ml-outputs/social-donation-lift/predictions.csv
Wrote: ../ml-outputs/social-donation-lift/recommended_windows.csv


,platform,day_of_week,post_hour,pred
0,Facebook,Friday,8,397945.733330
1,LinkedIn,Friday,7,374425.489987
2,Instagram,Wednesday,14,366868.828154
3,YouTube,Tuesday,1,257669.501055
4,Instagram,Monday,11,256500.455373
5,Instagram,Saturday,9,228131.345932
6,YouTube,Wednesday,12,221538.991623
7,Facebook,Friday,4,203770.142826
8,Facebook,Saturday,20,185371.078328
9,Facebook,Friday,18,183209.709053


## 5. Causal and Relationship Analysis

### Most Important Features
The top predictors of donation lift are typically: **platform** (Facebook and LinkedIn tend to drive higher direct donation values than TikTok or Twitter), **post_hour** and **day_of_week** (Friday morning tends to be optimal for donation-driving posts), **is_boosted** and **boost_budget_php** (paid amplification increases reach and therefore donations), and **follower_count_at_post_time** (audience size effect).

### Do These Relationships Make Theoretical Sense?
**Platform effects**: Facebook and LinkedIn have older, higher-income audiences who are more likely to donate online — this is consistent with donor demographic research showing that older platforms have higher-value donor audiences.

**Timing effects**: Friday donations align with 'end-of-week reflection' behavior and proximity to weekend — periods when people make charitable decisions. This is theoretically plausible but also could reflect selection effects in when the organization posts its highest-effort content.

**Boost budget**: This relationship is partially causal (paid reach → more eyes → more donations) but also reflects organizational decision-making (staff boost posts they believe are high-quality).

### Can We Make Causal Claims?
**Very limited causal claims are defensible:**
- Posting timing has *some* causal component: posting at 9am vs. 9pm genuinely changes the population of users who see the post
- Boost budget has a *partial* causal effect: more money does increase reach
- Platform effects are NOT causal in an actionable sense: the organization cannot move all its content from Twitter to Facebook and expect the same effect (audience trust is built platform-specifically)

**Confounders we cannot control for:**
- Fundraising campaign context (posts during active campaigns generate more donations regardless of content quality)
- News cycles and external events affecting charitable giving
- Post quality (creativity, emotional resonance) is not captured in metadata features

### Key Insight for the Organization
The model reveals that **timing and platform distribution** of posting effort are the most controllable levers for maximizing donation lift. The recommended posting windows (Friday morning on Facebook, Friday morning on LinkedIn) represent actionable guidance the team can implement immediately without changing content strategy.

### Limitations
- Donation attribution to specific posts is inherently imprecise — not all donations can be cleanly attributed to a single post
- Donation values are highly variable; R² is expected to be low
- The model reflects historical patterns that may shift with audience growth or platform algorithm changes
- Small holdout set makes evaluation estimates uncertain

## 6. Deployment Notes

### Web Application Integration

**Public Social Media Insights Page** (`/insights`) — Donation Impact tab:
- File: `frontend/src/pages/InsightsPage.tsx`
- Shows 'Best Times to Post for Donation Impact' section with recommended windows per platform
- Displays predicted donation value (in PHP) as ranking metric for each window
- The public can see these recommendations, demonstrating the organization's data-driven approach

### Data Flow
1. This notebook runs and exports:
   - `ml-outputs/social-donation-lift/predictions.csv` — holdout predictions
   - `ml-outputs/social-donation-lift/recommended_windows.csv` — top posting windows
2. Recommended windows are converted to `frontend/src/data/ml/socialDonationLift.ts`
3. React app imports this data file directly

### Exported CSV Schema
**predictions.csv:**
```
post_id                  : int   — Post identifier
post_url                 : str
created_at               : datetime
y_true                   : float — Actual donation value (PHP)
y_pred                   : float — Predicted donation value (PHP)
abs_error                : float — |y_true - y_pred|
```
**recommended_windows.csv:**
```
platform                 : str
day_of_week              : str
post_hour                : int
pred                     : float — Mean predicted donation value for this window
```

### Note on Presentation
Predicted donation values in the UI are presented as relative rankings rather than absolute forecasts, with appropriate uncertainty language.